# Multi-Model Robustness: Corrected Power Law Exponent

Tests whether the ~0.75 coherence decay exponent is stable across different probe models, and whether the human/AI divergence replicates beyond Mistral-7B.

**Models tested here**:
- GPT-2 (124M) — small, English-only, older architecture
- GPT-2 Medium (355M) — same family, 3x larger
- Llama-3-8B (8B, 4-bit) — independent 8B model, different training data/architecture

**Reference** (from RAID v7, already computed):
- Mistral-7B: human α = -0.75, AI α = -1.97

**Key question**: Does the human/AI gap replicate with Llama-3-8B? GPT-2 models were too weak to detect the gap (AI r values ~0.55). Llama at 8B should have the capacity.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
# === Configuration ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

MODELS = {
    'gpt2': {
        'name': 'GPT-2 (124M)',
        'hf_name': 'gpt2',
        'use_4bit': False,
        'color': '#2ca02c',
    },
    'gpt2-medium': {
        'name': 'GPT-2 Medium (355M)',
        'hf_name': 'gpt2-medium',
        'use_4bit': False,
        'color': '#ff7f0e',
    },
    'llama3-8b': {
        'name': 'Llama-3-8B',
        'hf_name': 'meta-llama/Meta-Llama-3-8B',
        'use_4bit': True,
        'color': '#9467bd',
    },
}

# Mistral reference (from v7, not re-run)
MISTRAL_REF = {
    'human': {'slope': -0.75, 'r': -0.87},
    'ai': {'slope': -1.97, 'r': -0.95},
}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    BASE_DIR = Path("/content/drive/MyDrive/LRTIA/Results/RAID_multimodel_finegrain")
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR = Path("../results/RAID_multimodel_finegrain")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MAX_CONTEXT = 100
TARGET_LEN = 30
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
N_DOCS = 150
RANDOM_SEED = 42
DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']

print(f"Models to run: {', '.join(m['name'] for m in MODELS.values())}")
print(f"Mistral-7B reference: human α={MISTRAL_REF['human']['slope']}, AI α={MISTRAL_REF['ai']['slope']}")
print(f"Max context: {MAX_CONTEXT}, Target length: {TARGET_LEN}")

In [ ]:
# === Load RAID corpus (same sample for all models) ===
corpus_all = []
with open(DATA_DIR / "raid_corpus.jsonl") as f:
    for line in f:
        corpus_all.append(json.loads(line))

rng = np.random.RandomState(RANDOM_SEED)
sample = []
for pop in ['human', 'ai']:
    pop_docs = [d for d in corpus_all if d['population'] == pop and len(d['text'].split()) >= 250]
    for domain in DOMAINS:
        pool = [d for d in pop_docs if d['domain'] == domain]
        n = min(len(pool), N_DOCS // len(DOMAINS) + 1)
        if n > 0:
            sample.extend(rng.choice(pool, size=n, replace=False))

print(f"Selected {len(sample)} documents")
print(f"  Human: {sum(1 for d in sample if d['population'] == 'human')}")
print(f"  AI: {sum(1 for d in sample if d['population'] == 'ai')}")

In [ ]:
# === Analysis functions ===
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

print("Analysis functions defined")

In [ ]:
# === Main loop: process each model sequentially ===
# Load model, run pipeline, unload, repeat.

all_model_results = {}

for model_key, model_info in MODELS.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {model_info['name']} ({model_info['hf_name']})")
    print(f"{'='*60}")

    # Check cache
    intact_path = BASE_DIR / f"{model_key}_intact_v1.json"
    shuffled_path = BASE_DIR / f"{model_key}_shuffled_v1.json"

    if intact_path.exists() and shuffled_path.exists():
        with open(intact_path) as f:
            all_intact = json.load(f)
        with open(shuffled_path) as f:
            all_shuffled = json.load(f)
        print(f"  Loaded {len(all_intact)} intact + {len(all_shuffled)} shuffled from cache")
        all_model_results[model_key] = {'intact': all_intact, 'shuffled': all_shuffled}
        continue

    # Load model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_info['hf_name'])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if model_info['use_4bit']:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_info['hf_name'], quantization_config=bnb_config, device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_info['hf_name'], torch_dtype=torch.float16 if device == "cuda" else torch.float32
        )
        model = model.to(device)
    model.eval()
    print(f"  Model loaded on {device}")

    # Define compute functions using this model
    @torch.no_grad()
    def compute_ppl(token_ids, target_start, target_end):
        if target_start >= target_end - 1:
            return float('inf')
        input_ids = torch.tensor([token_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        total_loss = 0.0
        count = 0
        for i in range(target_start, target_end - 1):
            log_probs = torch.log_softmax(logits[i], dim=-1)
            total_loss += -log_probs[token_ids[i + 1]].item()
            count += 1
        del outputs, logits
        torch.cuda.empty_cache()
        return math.exp(total_loss / count) if count > 0 else float('inf')

    def compute_token_reveal_curve(full_ids, target_start, target_end,
                                    shuffled=False, rng_shuf=None):
        target_ids = full_ids[target_start:target_end]
        context_pool = list(full_ids[:target_start])
        if shuffled and rng_shuf is not None:
            context_pool = list(context_pool)
            rng_shuf.shuffle(context_pool)
        max_ctx = min(MAX_CONTEXT, len(context_pool))
        if max_ctx < 10:
            return None
        ppls, ctx_lengths = [], []
        for ctx_len in range(1, max_ctx + 1):
            ctx_tokens = context_pool[-ctx_len:]
            chunk = ctx_tokens + target_ids
            ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
            if not math.isinf(ppl):
                ppls.append(ppl)
                ctx_lengths.append(ctx_len)
        if len(ppls) < 10:
            return None
        return {'ctx_lengths': ctx_lengths, 'ppls': ppls}

    # Process documents
    all_intact = []
    all_shuffled = []
    rng_shuf = np.random.RandomState(RANDOM_SEED + 99)

    for doc in tqdm(sample, desc=model_info['name']):
        full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
        n = len(full_ids)
        for frac in TARGET_FRACTIONS:
            target_start = int(n * frac)
            target_end = min(target_start + TARGET_LEN, n)
            if target_start < MIN_CONTEXT_BEFORE_TARGET or target_end - target_start < 5:
                continue
            result = compute_token_reveal_curve(full_ids, target_start, target_end)
            if result is not None:
                result['doc_id'] = doc['doc_id']
                result['population'] = doc['population']
                result['target_frac'] = frac
                all_intact.append(result)
            result_s = compute_token_reveal_curve(full_ids, target_start, target_end,
                                                  shuffled=True, rng_shuf=rng_shuf)
            if result_s is not None:
                result_s['doc_id'] = doc['doc_id']
                result_s['population'] = doc['population']
                result_s['target_frac'] = frac
                all_shuffled.append(result_s)

    # Save
    with open(intact_path, 'w') as f:
        json.dump(all_intact, f)
    with open(shuffled_path, 'w') as f:
        json.dump(all_shuffled, f)
    print(f"  Computed {len(all_intact)} intact + {len(all_shuffled)} shuffled curves")

    all_model_results[model_key] = {'intact': all_intact, 'shuffled': all_shuffled}

    # Unload model to free memory
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  Model unloaded")

print("\n\nAll models processed!")

In [ ]:
# === Compute exponents for each model x population ===
model_fits = {}  # model_key -> {human: fit, ai: fit}

print(f"\n{'Model':<25} {'Population':<10} {'α':>8} {'r':>8} {'p':>10} {'N':>6}")
print("-" * 70)

for model_key, results in all_model_results.items():
    model_fits[model_key] = {}
    for pop in ['human', 'ai']:
        intact = [c for c in results['intact'] if c['population'] == pop]
        shuffled = [c for c in results['shuffled'] if c['population'] == pop]
        if len(intact) < 5 or len(shuffled) < 5:
            continue
        ip = compute_raw_ppl_curve(intact)
        sp = compute_raw_ppl_curve(shuffled)
        corr = -np.diff(ip) - (-np.diff(sp))
        fit = fit_power_law(corr)
        if fit:
            model_fits[model_key][pop] = {
                'slope': fit[0], 'r': fit[1], 'p': fit[2],
                'bc': fit[3], 'bm': fit[4], 'intercept': fit[5],
                'corrected_marg': corr, 'n': len(intact),
            }
            name = MODELS[model_key]['name']
            print(f"{name:<25} {pop:<10} {fit[0]:>8.3f} {fit[1]:>8.3f} {fit[2]:>10.4f} {len(intact):>6}")

In [ ]:
# === Figure 1: Multi-model comparison ===
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel A: Power law fits overlaid (human only) ---
ax = axes[0]
for model_key, fits in model_fits.items():
    if 'human' not in fits:
        continue
    f = fits['human']
    info = MODELS[model_key]
    ax.plot(f['bc'], f['bm'], 'o-', color=info['color'], linewidth=2, markersize=6,
            label=f"{info['name']}: α={f['slope']:.2f}")
    fit_x = np.linspace(min(f['bc']), max(f['bc']), 100)
    ax.plot(fit_x, np.exp(f['intercept']) * fit_x**f['slope'], '--',
            color=info['color'], alpha=0.4)
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal Benefit', fontsize=12)
ax.set_title('A. Human Text: Power Law Across Models', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# --- Panel B: Bar chart — include Mistral reference ---
ax = axes[1]

# Build entries: computed models + Mistral reference
bar_entries = []
for model_key in MODELS:
    if model_key not in model_fits:
        continue
    info = MODELS[model_key]
    for pop in ['human', 'ai']:
        if pop in model_fits[model_key]:
            f = model_fits[model_key][pop]
            bar_entries.append((f"{info['name']}\n({'Human' if pop == 'human' else 'AI'})",
                                f['slope'], f['r'],
                                info['color'] if pop == 'human' else '#d62728'))

# Add Mistral reference
bar_entries.append(('Mistral-7B\n(Human, ref)', MISTRAL_REF['human']['slope'],
                    MISTRAL_REF['human']['r'], '#1f77b4'))
bar_entries.append(('Mistral-7B\n(AI, ref)', MISTRAL_REF['ai']['slope'],
                    MISTRAL_REF['ai']['r'], '#d62728'))

labels = [e[0] for e in bar_entries]
values = [e[1] for e in bar_entries]
rs = [e[2] for e in bar_entries]
colors = [e[3] for e in bar_entries]

ax.bar(range(len(bar_entries)), values, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(bar_entries)))
ax.set_xticklabels(labels, fontsize=7, rotation=35, ha='right')
ax.set_ylabel('Power Law Exponent (α)', fontsize=12)
ax.set_title('B. Exponents by Model and Population', fontweight='bold')
ax.axhline(-0.77, color='gray', linestyle=':', alpha=0.5, label='Anderson & Schooler')
ax.grid(True, alpha=0.2, axis='y')
ax.legend(fontsize=9)

for i, (val, r_val) in enumerate(zip(values, rs)):
    ax.text(i, val - 0.07, f'{val:.2f}\n(r={r_val:.2f})',
            ha='center', fontsize=7, fontweight='bold')

# --- Panel C: Corrected marginals overlaid (human, computed models) ---
ax = axes[2]
for model_key, fits in model_fits.items():
    if 'human' not in fits:
        continue
    f = fits['human']
    info = MODELS[model_key]
    ax.plot(common_x[1:], uniform_filter1d(f['corrected_marg'], 5),
            '-', color=info['color'], linewidth=2, label=info['name'])
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal', fontsize=12)
ax.set_title('C. Coherence Signal: Human Text Across Models', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.suptitle('Model Robustness: Corrected Coherence Decay (124M → 7B parameters)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_multimodel_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\n" + "="*60)
print("MULTI-MODEL SUMMARY")
print("="*60)
for model_key in MODELS:
    if model_key not in model_fits:
        continue
    info = MODELS[model_key]
    for pop in ['human', 'ai']:
        if pop in model_fits[model_key]:
            f = model_fits[model_key][pop]
            print(f"  {info['name']:<25} {pop:<8}: α = {f['slope']:.3f} (r = {f['r']:.3f})")
print(f"  {'Mistral-7B (ref)':<25} {'human':<8}: α = {MISTRAL_REF['human']['slope']:.3f} (r = {MISTRAL_REF['human']['r']:.3f})")
print(f"  {'Mistral-7B (ref)':<25} {'ai':<8}: α = {MISTRAL_REF['ai']['slope']:.3f} (r = {MISTRAL_REF['ai']['r']:.3f})")

# Human exponent stability
human_exps = [model_fits[k]['human']['slope'] for k in model_fits if 'human' in model_fits[k]]
human_exps.append(MISTRAL_REF['human']['slope'])
print(f"\n  Human exponent range (all 3 models): [{min(human_exps):.3f}, {max(human_exps):.3f}]")
print(f"  Human exponent mean:  {np.mean(human_exps):.3f} ± {np.std(human_exps):.3f}")